In [7]:
import os, sys, logging, importlib, math, time, subprocess
import rasterio, affine, pycountry

import geopandas as gpd
import pandas as pd

from zipfile import ZipFile
from affine import Affine
from rasterio import features
from rasterio.mask import mask
from rasterio.features import rasterize

import GEP

'''prints the time along with the message'''
def tPrint(s):
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

# Mapping electrification
Code to download results from S3

We need to convert certain fields in the results to a raster; these fields need to be calculated 

1. Population already electrified 2030  
    a. Rasterize field **ElecPop**
2. New population electrified by grid 2030  
    a. Filter data where **FinalElecCode2030** == 1.0  
    b. Set **NewConnections2025** = 0 where **FinalElecCode2025** == 99.0  
    c. Combine **NewConnections2030** with **NewConnections2025**  
3. New population electrified by off grid 2030  
    a. Filter data where **FinalElecCode2030** != 1.0  
    b. Set **NewConnections2025** = 0 where **FinalElecCode2025** != **FinalElecCode2030**  
    c. Combine **NewConnections2030** with **NewConnections2025**  

In [3]:
# delete all existing rasterized results
'''all_rasters = []
for root, dirs, files in os.walk('/media/gost/DATA1/GEP/Clusters'):
    for f in files:
        if "rasterized_gep" in f:
            all_rasters.append(os.path.join(root, f))
for f in all_rasters:
    os.remove(f)'''

'all_rasters = []\nfor root, dirs, files in os.walk(\'/media/gost/DATA1/GEP/Clusters\'):\n    for f in files:\n        if "rasterized_gep" in f:\n            all_rasters.append(os.path.join(root, f))\nfor f in all_rasters:\n    os.remove(f)'

In [4]:
resolution = 0.01 #Resolution of output raster in degrees
elecRate = 0.90   #settlement electrification rate to be considered electrified
year = 2030       #year of analysis 
scenario = "0_0_0_0_1_0" 
base_folder = r'C:\WBG\Work\GEP\Climate-2022'
data_folder = os.path.join(base_folder, 'Data')
results_folder = os.path.join(base_folder, 'Results')
rasterized_results_folder = os.path.join(results_folder, 'Rasterized_Scenarios')
scenario_base_folder =  os.path.join(data_folder, 'Scenarios')
scenario_folder = os.path.join(scenario_base_folder, scenario)
summaries_folder = os.path.join(data_folder, 'Scenario_Summaries')
zip_folder =       os.path.join(scenario_base_folder, 'zipfiles')
spatial_folder =   os.path.join(data_folder, 'SpatialData')


In [5]:
scenario_file = os.path.join(scenario_base_folder, "3-0_0_0_0_1_0_Global_merge.csv")
in_scenario = pd.read_csv(scenario_file)
in_scenario['Country'] = in_scenario['Country'].str.lower() 

GEP_countries = [x.lower() for x in in_scenario.Country.unique()]
countries = [x for x in os.listdir(spatial_folder) if os.path.isdir(os.path.join(spatial_folder, x))]

C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\3265971469.py:2: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  in_scenario = pd.read_csv(scenario_file)


In [116]:
def rasterize_results(inD, outFile,
        field='FinalElecCode2030', res=0.1, dtype='uint8', merge_alg='REPLACE', sort_ascending=False):
    """_summary_

    Parameters
    ----------
    inD : gpd.GeoDataFrame
        input settlements layer with columns to rasterize
    outFile : string
        path to create output raster
    field : str, optional
        field to rasterize, by default 'FinalElecCode2030'
    res : float, optional
        resolution to create output raster file, in crs of inD, by default 0.1
    dtype : str, optional
        datatype of output raster, by default 'uint8'
    merge_alg : str, optional
        how to merge new results, by default 'REPLACE', other options are 'ADD'
    sort_ascending : bool, optional
        sort inD dataset by [field] in ascending order, by default False
    """

    if merge_alg == "REPLACE":
        merge_alg = rasterio.enums.MergeAlg.replace
    else:
        merge_alg = rasterio.enums.MergeAlg.add

    # create metadata
    bounds = inD.total_bounds
    # calculate height and width from resolution
    width = math.ceil((bounds[2] - bounds[0]) / res)
    height = math.ceil((bounds[3] - bounds[1]) / res)

    cAffine = affine.Affine(res, 0, bounds[0], 0, res * -1, bounds[3])
    nTransform = cAffine #(res, 0, bounds[2], 0, res * -1, bounds[1])
    cMeta = {'count':1,
                'crs': inD.crs,
                'dtype':dtype,
                'affine':cAffine,
                'driver':'GTiff',
                'transform':nTransform,
                'height':height,
                'width':width,
                'nodata': None}
    inD = inD.sort_values(by=[field], ascending=sort_ascending)
    inD[field] = inD[field].astype(dtype)
    shapes = ((row.geometry, row[field]) for idx, row in inD.iterrows())
    with rasterio.open(outFile, 'w', **cMeta) as out:
        burned = features.rasterize(shapes=shapes,
                                    fill=0,
                                    all_touched=True,
                                    out_shape=(cMeta['height'], cMeta['width']),
                                    transform=out.transform,
                                    merge_alg=merge_alg)
        burned = burned.astype(cMeta['dtype'])
        out.write_band(1, burned)
    return(burned)

In [117]:
res = 0.05

out_raster = os.path.join(rasterized_results_folder, f'rasterized_gep_{scenario}_{iso2}_res{res}.tif')
out_raster_pop = os.path.join(rasterized_results_folder, f'rasterized_pop_{scenario}_{iso2}_res{res}.tif')
out_raster_elecpop = os.path.join(rasterized_results_folder, f'rasterized_elecpop_{scenario}_{iso2}_res{res}.tif')
out_raster_perelec = os.path.join(rasterized_results_folder, f'rasterized_perelec_{scenario}_{iso2}_res{res}.tif')
out_raster_elec = os.path.join(rasterized_results_folder, f'rasterized_electrified_{scenario}_{iso2}_res{res}.tif')
    

In [118]:
res = 0.05


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\3079216699.py:6: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\1183352324.py:1: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


In [98]:
res = 0.1
#rasterize_results(inD, out_raster, field='FinalElecCode2030', res=res, dtype='uint8', merge_alg='REPLACE')
burned_res = rasterize_results(inD, out_raster_elec, field='PercentElec',  res=0.01, dtype='uint8', merge_alg='REPLACE')
        

In [127]:
sel_cols = ['id','Country', 'IsUrban','NightLights', 'Pop','ElecPopCalib','FinalElecCode2030']
res = 0.1
for country in countries:
    iso2 = country[:2]
    country_search = pycountry.countries.get(alpha_2=iso2)
    country_name = country_search.name.lower().replace("-", " ")
    if iso2 == 'cd':
        country_name = 'democratic republic of congo'
    elif iso2 == 'cg':
        country_name = 'republic of congo'
    elif iso2 == 'ci':
        country_name = "ivory coast"
    elif iso2 == 'fm':
        country_name = "micronesia"
    elif iso2 == 'tz':
        country_name = "tanzania"
    elif iso2 == 'ni':
        country_name = "nicaraqua"
    country_maps = country_name in GEP_countries

    out_raster = os.path.join(rasterized_results_folder, f'{iso2}_eleccode_{scenario}_res{res}.tif')
    out_raster_pop = os.path.join(rasterized_results_folder, f'{iso2}_pop_{scenario}_{iso2}_res{res}.tif')
    out_raster_elecpop = os.path.join(rasterized_results_folder, f'{iso2}_elecpop_{scenario}_{iso2}_res{res}.tif')
    out_raster_perelec = os.path.join(rasterized_results_folder, f'{iso2}_perelec_{scenario}_{iso2}_res{res}.tif')
    out_raster_elec = os.path.join(rasterized_results_folder, f'{iso2}_ electrified_{scenario}_{iso2}_res{res}.tif')
    country_shape_folder = os.path.join(spatial_folder, country)
    
    create_maps = False
    for x in [out_raster, out_raster_pop, out_raster_elecpop, out_raster_perelec]:
        if not os.path.exists(x):
            create_maps = True
            break

    if country_maps and create_maps:
        tPrint(f'{iso2} {country_name} - {country_maps}')
        shape_file = [os.path.join(country_shape_folder, f) for f in os.listdir(country_shape_folder) if f.endswith('.shp')][0]        
        shapeD = gpd.read_file(shape_file)
        shapeD.set_index("id", inplace=True)
        # select out the scenario results from the global _ file
        cur_scenario = in_scenario.loc[in_scenario['Country'] == country_name,sel_cols]
        cur_scenario = cur_scenario.set_index('id')
        inD = shapeD.join(cur_scenario, lsuffix='l_', rsuffix='')
        inD = inD.sort_values(by='ElecPopCalib')
        
        inD['Pop'] = inD['Pop'].fillna(1)
        inD['FinalElecCode2030'] = inD['FinalElecCode2030'].fillna(0)        
        inD['ElecPopCalib'] = inD['ElecPopCalib'].fillna(0)
        
        
        inD['PercentElec'] = inD.apply(lambda x: int(x['ElecPopCalib'] / x['Pop'] * 100), axis=1)
        elec_code_res = rasterize_results(inD, out_raster, field='FinalElecCode2030', res=res, dtype='uint8', merge_alg='REPLACE', sort_ascending=False)
        #percent_elec = rasterize_results(inD, out_raster_elec, field='PercentElec',  res=res, dtype='uint8', merge_alg='REPLACE', sort_ascending=True)
        pop_res = rasterize_results(inD, out_raster_pop, field='Pop',  res=res, dtype='float64', merge_alg='ADD')
        elec_pop_res = rasterize_results(inD, out_raster_elecpop, field='ElecPopCalib',  res=res, dtype='float64', merge_alg='ADD') 
        elec_rate = elec_pop_res/pop_res
        cMeta = rasterio.open(out_raster_elecpop).meta.copy()
        with rasterio.open(out_raster_perelec, 'w', **cMeta) as out:
            out.write_band(1, elec_rate)

12:39:54	bf burkina faso - True


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\35647315.py:55: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


12:46:06	bi burundi - True


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\35647315.py:55: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


12:48:08	bj benin - True


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\35647315.py:55: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


... (92 outputs left out)

16:57:42	zm zambia - True


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\35647315.py:55: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


17:06:36	zw zimbabwe - True


C:\Users\WB411133\AppData\Local\Temp\ipykernel_6208\35647315.py:55: RuntimeWarning: invalid value encountered in divide
  elec_rate = elec_pop_res/pop_res


In [4]:
importlib.reload(GEP)
# Loop through all countries and rasterize
minPop = 100
for country in processedCountries:
    xx_so = GEP.gep_results(country[:4], 
                scenariosFolder=scenario_base,
                clustersFolder = cluster_folder,
                scenario=scenario)
    break
    outFile = os.path.join(raster_folder, f"{country[:2]}_rasterized_gep_{resolution}_{scenario}_{year}_gt{minPop}.tif")    
    if not os.path.exists(outFile):
        tPrint(f"Processing {country}")    
        res = xx_so.join_results()
        res = res.loc[res['Pop2030'] > minPop,]
        # Identify already electrified settlements and set value to 98
        res['already_electrified'] = (res['ElecPopCalib'] / res['Pop2020']) > elecRate
        res.loc[res['already_electrified'] == 1, 'FinalElecCode%s' % year] = 98
        xx_so.rasterize_results(res, outFile, res=resolution, field='FinalElecCode%s' % year)
    else:
        tPrint(f"{country} already rasterized")    

In [ ]:
### Run a series of GDAL commands to create No Data rasters and stack them
command_file = os.path.join(vrt_folder, f'create_vrt_{resolution}_{scenario}_{year}.sh')
with open(command_file, 'w') as vrt_file:
    vrt_file.write('#!/bin/sh \n')
    # Convert 0 in data to no data
    outFolder = raster_folder
    allImages = []
    for root, folders, files in os.walk(outFolder):
        for f in files:
            if "%s_%s_%s_gt%s.tif" % (resolution, scenario, year, minPop) in f and f.endswith(".tif"):
                allImages.append(os.path.join(root, f))

    for inFile in allImages:
        outFile = inFile.replace(".tif", ".tif")
        vrt_file.write(f'gdal_translate -of GTiff -a_nodata 0 {inFile} {outFile}')
        vrt_file.write("\n")
    vrt_file.write(f'find {raster_folder} -name "*{resolution}_{scenario}_{year}_gt{minPop}_noData.tif" > tifFiles.txt')
    vrt_file.write("\n")
    vrt_file.write(f"gdalbuildvrt -input_file_list tifFiles.txt {vrt_folder}/combined_gep_rasterized_{resolution}_{scenario}_{year}.vrt")
    vrt_file.write("\n")
subprocess.call(command_file)

# Mapping annual electrification breakdown
The GEP results contain information describing the electrification status in 2020, 2025, and 2030 (ElecStart,ElecStatusIn2025, ElecStatusIn2030). Based on these we can separate the dataset into three groups:

1. Electrified
2. Electrified first - electrified between 2020 and 2025
3. Electrified second - electrified between 2025 and 2030

For the second and third groups, we can approximate the year they were electrified in that range based on other variables. In the example below, we use the distance to MV lines, under the assumption that places closer to MV lines will be electrified first.

The code works by take the two groups separately, sorting them according the the predictive variable, and then assigning the year of electrification sequentially.

In [6]:
def get_years(shp, start_year, end_year):
    '''    
        Get a list of numbers between start_year and end_year of length shp    
    '''
    n_years = round(shp/(end_year - start_year + 1))
    yr = [x for x in range(start_year, end_year + 1) for _ in range(n_years)]
    if len(yr) > shp:
        return(yr[:shp])
    while len(yr) < shp:
        yr.append(end_year)        
    return(yr)

get_years(10, 2, 6)

[2, 2, 3, 3, 4, 4, 5, 5, 6, 6]

In [7]:
# Get a list of all countries for the selected scenario
importlib.reload(GEP)
scenario = "0_0_0_0_1_0"
scenario_folder = os.path.join(scenario_base, f'{scenario}')
# Identify countries to process
zipFiles = []
processedCountries = []
for root, folders, files in os.walk(scenario_folder):    
    processedCountries.append([f for f in folders])
    for f in files:
        if f[-4:] == ".csv":
            zipFiles.append(f[:-4])
processedCountries = [item[:4] for item in zipFiles]
zipFiles #list of country/scenarios to process

['gh-2-0_0_0_0_1_0',
 'mw-2-0_0_0_0_1_0',
 'ao-2-0_0_0_0_1_0',
 'bd-2-0_0_0_0_1_0',
 'bf-2-0_0_0_0_1_0',
 'bi-2-0_0_0_0_1_0',
 'bj-2-0_0_0_0_1_0',
 'bw-2-0_0_0_0_1_0',
 'cd-2-0_0_0_0_1_0',
 'cf-2-0_0_0_0_1_0',
 'cg-2-0_0_0_0_1_0',
 'ci-2-0_0_0_0_1_0',
 'cm-2-0_0_0_0_1_0',
 'dj-2-0_0_0_0_1_0',
 'er-2-0_0_0_0_1_0',
 'et-2-0_0_0_0_1_0',
 'fm-2-0_0_0_0_1_0',
 'ga-2-0_0_0_0_1_0',
 'gm-2-0_0_0_0_1_0',
 'gn-2-0_0_0_0_1_0',
 'gq-2-0_0_0_0_1_0',
 'gw-2-0_0_0_0_1_0',
 'hn-2-0_0_0_0_1_0',
 'ht-2-0_0_0_0_1_0',
 'ke-2-0_0_0_0_1_0',
 'kh-2-0_0_0_0_1_0',
 'km-2-0_0_0_0_1_0',
 'lr-2-0_0_0_0_1_0',
 'ls-2-0_0_0_0_1_0',
 'mg-2-0_0_0_0_1_0',
 'ml-2-0_0_0_0_1_0',
 'mm-2-0_0_0_0_1_0',
 'mr-2-0_0_0_0_1_0',
 'mz-2-0_0_0_0_1_0',
 'na-2-0_0_0_0_1_0',
 'ne-2-0_0_0_0_1_0',
 'ng-2-0_0_0_0_1_0',
 'ni-2-0_0_0_0_1_0',
 'pg-2-0_0_0_0_1_0',
 'pk-2-0_0_0_0_1_0',
 'rw-2-0_0_0_0_1_0',
 'sb-2-0_0_0_0_1_0',
 'sd-2-0_0_0_0_1_0',
 'sl-2-0_0_0_0_1_0',
 'sn-2-0_0_0_0_1_0',
 'so-2-0_0_0_0_1_0',
 'ss-2-0_0_0_0_1_0',
 'st-2-0_0_0_

In [8]:
# This will run for a single country, but could be looped over as is done above
country = 'zw-2'
sort_column = 'CurrentMVLineDist'

In [11]:
importlib.reload(GEP)
out_folder = '/media/gost/DATA1/GEP/rasterized_yearly'
for country in ['zw-2-0_0_0_0_1_0']: #zipFiles:    
    outFile = os.path.join(out_folder, f"{country}_electrification_evolution_{sort_column}.tif")
    popFile = os.path.join(out_folder, f"{country}_Pop2030_{sort_column}.tif")
    if True: #not os.path.exists(outFile):
        tPrint(country)
        xx_so = GEP.gep_results(country[:4], 
                    scenariosFolder=scenario_base,
                    clustersFolder = cluster_folder,
                    scenario=scenario)
    
        res = xx_so.join_results()        
        res['perElec'] = (res['ElecPopCalib'] / res['Pop2020'])
        sel_res = res.loc[:,['id','ElecStart','ElecStatusIn2025','ElecStatusIn2030','CurrentMVLineDist','perElec','Pop2030','geometry']]
        # Set the default value (already electrified)
        sel_res['year_elec'] = 2018
        # Select features that are electrified between 2020 and 2025
        first = sel_res.loc[(sel_res['ElecStart'] == 0) & (sel_res['ElecStatusIn2025'] == 1)]
        first = first.sort_values([sort_column]) #Sort by the predictive column
        first['year_elec'] = get_years(first.shape[0], 2019, 2024)
        # Select features that are electrified between 2025 and 2030
        second = sel_res.loc[(sel_res['ElecStatusIn2025'] == 0) & (sel_res['ElecStatusIn2030'] == 1)]
        second = second.sort_values([sort_column])
        second['year_elec'] = get_years(second.shape[0], 2025, 2030)

        sel_res.loc[first.index, 'year_elec'] = first['year_elec']
        sel_res.loc[second.index, 'year_elec'] = second['year_elec']

        sel_res['year_elec'] = sel_res['year_elec'] - sel_res['year_elec'].min() + 1

        #Before rasterizing, sort by year of electrification; I am not convinced this does anything
        sel_res = sel_res.sort_values(['year_elec'], ascending="year_elec")
        xx_so.rasterize_results(sel_res, outFile, res=resolution, field='year_elec')
        xx_so.rasterize_results(sel_res, popFile, res=resolution, field='Pop2030', dtype="float32", merge_alg="ADD")

15:57:32	zw-2-0_0_0_0_1_0


In [13]:
res.columns

Index(['id', 'geometry', 'Unnamed: 0', 'AgriDemand', 'CommercialDemand',
       'Country', 'CurrentHVLineDist', 'CurrentMVLineDist', 'EducationDemand',
       'GHI', 'GridCellArea', 'HealthDemand', 'HydropowerDist', 'IsUrban',
       'NightLights', 'PerCapitaDemand', 'PlannedHVLineDist',
       'PlannedMVLineDist', 'Pop', 'ResidentialDemandTierCustom', 'RoadDist',
       'SubstationDist', 'TransformerDist', 'TravelHours', 'X_deg', 'Y_deg',
       'Admin1', 'Cat_1', 'Cat_2', 'Cat_3', 'Unc', 'Prim', 'Sec',
       'GridPenalty', 'WindCF', 'PopStartYear', 'ElecPopCalib', 'Pop2025',
       'Pop2030', 'Pop2020', 'ElecStart', 'GridDistCalibElec',
       'FinalElecCode2020', 'Commercial_Multiplier', 'MVConnectDist',
       'NewConnections2025', 'NumPeoplePerHH', 'EnergyPerSettlement2025',
       'TotalEnergyPerCell', 'Tier', 'MG_PV_Hybrid2025', 'MG_Wind_Hybrid2025',
       'MG_Hydro2025', 'SA_PV2025', 'Grid2025', 'MinGridDist2025',
       'ElectrificationOrder2025', 'MinimumOverall2025',
     

In [ ]:
raster_folder = '/media/gost/DATA1/GEP/rasterized_yearly'

### Run a series of GDAL commands to create No Data rasters and stack them
command_file = os.path.join(vrt_folder, f'create_vrt_{sort_column}_yearly.sh')
with open(command_file, 'w') as vrt_file:
    vrt_file.write('#!/bin/sh \n')
    # Convert 0 in data to no data
    outFolder = raster_folder
    allImages = []
    for root, folders, files in os.walk(out_folder):
        for f in files:
            if f.endswith(f"{sort_column}.tif"):
                allImages.append(os.path.join(root, f))

    for inFile in allImages:
        outFile = inFile.replace(".tif", ".tif")
        vrt_file.write(f'gdal_translate -of GTiff -a_nodata 0 {inFile} {outFile}')
        vrt_file.write("\n")
    vrt_file.write(f'find {raster_folder} -name "*.tif" > tifFiles.txt')
    vrt_file.write("\n")
    vrt_file.write(f"gdalbuildvrt -input_file_list tifFiles.txt {vrt_folder}/combined_gep_{sort_column}_yearly.vrt")
    vrt_file.write("\n")
subprocess.call(command_file)